# Concept Direction Analysis

This notebook loads preserved direct-op parity reports, upstream-only reference-graph sanity reports, and notebook debug artifacts for the preserved `gemma-3-1b-it` runs.

Typical artifact generation flow:

```bash
export IT_REPO_DIR=${HOME}/repos/interpretune
export IT_VENV_BASE=/mnt/cache/${USER}/.venvs
export IT_TARGET_VENV=it_latest

cd "${IT_REPO_DIR}" && \
  IT_RUN_STANDALONE_TESTS=1 \
  IT_PARITY_PRESERVE_ARTIFACTS=1 \
  IT_PARITY_PRESERVE_ARTIFACT_DIR="${IT_REPO_DIR}/tests/nb_experiments/concept_direction/analysis/artifacts" \
  "${IT_VENV_BASE}/${IT_TARGET_VENV}/bin/python" -m pytest \
  tests/nb_experiments/concept_direction/test_concept_direction_backend_parity.py \
  -k 'gemma3_1b_it_concept_direction_paths or gemma3_1b_it_bat_reference_graph_sanity' -vv
```

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path
from pprint import pprint

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        has_project = (candidate / "pyproject.toml").exists()
        has_tests = (candidate / "tests").exists()
        if has_project and has_tests:
            return candidate
    raise RuntimeError(f"Unable to locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

nb_ui_utils = importlib.import_module("src.it_examples.utils.nb_ui_utils")
display_top_features_comparison = nb_ui_utils.display_top_features_comparison

analysis_helpers = importlib.import_module("tests.nb_experiments.concept_direction.analysis.concept_direction_analysis")
analysis_helpers = importlib.reload(analysis_helpers)

ARTIFACT_GENERATION_ENV = analysis_helpers.ARTIFACT_GENERATION_ENV
PRESERVE_ARTIFACT_DIR_ENV = analysis_helpers.PRESERVE_ARTIFACT_DIR_ENV
discover_concept_direction_artifacts = analysis_helpers.discover_concept_direction_artifacts
load_concept_direction_parity_report = analysis_helpers.load_concept_direction_parity_report
load_concept_direction_pipeline_state_artifacts = analysis_helpers.load_concept_direction_pipeline_state_artifacts
load_concept_direction_reference_graph_report = analysis_helpers.load_concept_direction_reference_graph_report
resolve_concept_direction_parity_output_dir = analysis_helpers.resolve_concept_direction_parity_output_dir

In [ ]:
ARTIFACT_SELECTION_OVERRIDES = {
    "parity_reports": None,
    "reference_reports": None,
    "notebook_artifacts": None,
}


def _parse_selection_names(raw_value: str | None) -> tuple[str, ...] | None:
    if raw_value is None:
        return None
    names = tuple(sorted({item.strip() for item in raw_value.split(",") if item.strip()}))
    return names or None


def _resolve_selection_names(key: str, env_name: str) -> tuple[str, ...] | None:
    override = ARTIFACT_SELECTION_OVERRIDES.get(key)
    if override is not None:
        return tuple(str(value) for value in override)
    return _parse_selection_names(os.environ.get(env_name))


ARTIFACT_SELECTION = {
    "parity_reports": _resolve_selection_names(
        "parity_reports",
        "IT_CONCEPT_DIRECTION_PARITY_REPORTS",
    ),
    "reference_reports": _resolve_selection_names(
        "reference_reports",
        "IT_CONCEPT_DIRECTION_REFERENCE_REPORTS",
    ),
    "notebook_artifacts": _resolve_selection_names(
        "notebook_artifacts",
        "IT_CONCEPT_DIRECTION_NOTEBOOK_ARTIFACTS",
    ),
}

artifact_index = discover_concept_direction_artifacts(selection=ARTIFACT_SELECTION)
artifact_root = artifact_index["artifact_root"]
report_paths = artifact_index["report_paths"]
reference_report_paths = artifact_index["reference_report_paths"]
notebook_pipeline_state_paths = artifact_index["notebook_pipeline_state_paths"]
pipeline_state_paths = artifact_index["pipeline_state_paths"]

artifact_index

In [ ]:
NEURONPEDIA_MODEL_ID = "gemma-3-1b-it"
NEURONPEDIA_SOURCE_SET = "gemmascope-2-transcoder-16k"


def _json_cell(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return value
    return json.dumps(value)


def _short_sha(value):
    if not isinstance(value, str):
        return value
    return value[:12]


def _head(values, size: int = 5):
    if not isinstance(values, list):
        return values
    return values[:size]


def _direction_payload(section_payload: dict) -> dict:
    if not isinstance(section_payload, dict):
        return {}
    if "geometry" in section_payload:
        return section_payload
    nested = section_payload.get("direction")
    return nested if isinstance(nested, dict) else {}


def _normalize_feature_rows(rows) -> list[tuple[int, int, int]]:
    normalized_rows: list[tuple[int, int, int]] = []
    for row in rows or []:
        if isinstance(row, (list, tuple)) and len(row) == 3:
            normalized_rows.append((int(row[0]), int(row[1]), int(row[2])))
    return normalized_rows


def _graph_sections(report: dict) -> list[str]:
    return [
        key for key, payload in report.items() if isinstance(payload, dict) and isinstance(payload.get("graph"), dict)
    ]


def _reference_graph_labels(report: dict) -> list[str]:
    return [
        key
        for key, payload in report.items()
        if isinstance(payload, dict) and payload.get("top_feature_ids") is not None
    ]


def _graph_payload(report: dict, section: str) -> dict:
    payload = report.get(section, {})
    if isinstance(payload, dict) and isinstance(payload.get("graph"), dict):
        return payload["graph"]
    if isinstance(payload, dict):
        return payload
    return {}


def _feature_rows_from_payload(payload: dict) -> list[tuple[int, int, int]]:
    return _normalize_feature_rows(payload.get("top_feature_ids") or payload.get("top_features") or [])


def _feature_scores_from_payload(payload: dict) -> list[float]:
    scores = payload.get("top_feature_scores") or payload.get("feature_scores") or []
    return [float(value) for value in scores]


def _feature_partitions(payload: dict) -> tuple[list, list, list]:
    shared = payload.get("shared")
    if shared is None:
        shared = payload.get("shared_features") or payload.get("same_features") or []
    left_only = payload.get("left_only")
    if left_only is None:
        left_only = payload.get("left_only_features") or payload.get("embed_only_features") or []
    right_only = payload.get("right_only")
    if right_only is None:
        right_only = payload.get("right_only_features") or payload.get("store_only_features") or []
    return list(shared), list(left_only), list(right_only)


def _pairwise_feature_comparison(
    left_payload: dict,
    right_payload: dict,
    *,
    left_label: str,
    right_label: str,
) -> dict:
    return analysis_helpers.compare_top_feature_sets(
        _feature_rows_from_payload(left_payload),
        _feature_rows_from_payload(right_payload),
        left_scores=_feature_scores_from_payload(left_payload),
        right_scores=_feature_scores_from_payload(right_payload),
        left_label=left_label,
        right_label=right_label,
    ).to_dict()


def _artifact_surface(name: str) -> str | None:
    for surface in ("ohio", "orange", "bat"):
        if surface in name:
            return surface
    return None

In [ ]:
from html import escape as _html_escape

from IPython.display import HTML


def _visible_token_text(token_text) -> str:
    if token_text is None:
        return ""
    return str(token_text).replace("\r", "␍").replace("\n", "↵").replace("\t", "⇥")


def _token_debug_entry_html(row: dict) -> str:
    index = row.get("index")
    token_id = row.get("token_id")
    token_text = _visible_token_text(row.get("token_text"))
    marks = tuple(row.get("marks") or ())
    title_text = " | ".join(
        [
            f"index={index}",
            f"token_id={token_id}",
            f"marks={','.join(marks) if marks else 'none'}",
            f"raw={repr(row.get('token_text'))}",
        ]
    )
    style = [
        "display:inline-block",
        "margin:0.05rem 0.15rem 0.05rem 0",
        "padding:0.08rem 0.24rem",
        "border:1px solid #cbd5e1",
        "border-radius:0.35rem",
        "background:#f8fafc",
        "font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace",
        "font-size:0.75rem",
        "white-space:pre",
    ]
    if "cache_answer" in marks or "answer" in marks:
        style.extend(["background:#fff3cd", "border-color:#b7791f"])
    elif "context" in marks:
        style.extend(["background:#dcfce7", "border-color:#15803d"])
    elif "probe" in marks:
        style.extend(["background:#dbeafe", "border-color:#2563eb"])
    label = token_text if token_text else "∅"
    style_text = "; ".join(style)
    return f'<span title="{_html_escape(title_text, quote=True)}" style="{style_text}">{_html_escape(label)}</span>'


def _format_token_debug_html(rows) -> str:
    normalized_rows = [row for row in rows or [] if isinstance(row, dict)]
    if not normalized_rows:
        return ""
    rendered_rows = "".join(_token_debug_entry_html(row) for row in normalized_rows)
    return f'<div style="line-height:1.5; max-width:96rem; white-space:normal">{rendered_rows}</div>'


def display_html_frame(frame: pd.DataFrame) -> None:
    html_columns = tuple(frame.attrs.get("html_columns", ()))
    safe_frame = frame.copy()
    for column in safe_frame.columns:
        if column in html_columns:
            continue
        safe_frame[column] = safe_frame[column].map(
            lambda value: _html_escape(value) if isinstance(value, str) else value
        )
    display(HTML(safe_frame.to_html(index=False, escape=False)))


def display_full_frame(frame: pd.DataFrame) -> None:
    if frame.attrs.get("html_columns"):
        display_html_frame(frame)
        return
    with pd.option_context(
        "display.max_columns",
        None,
        "display.max_colwidth",
        None,
        "display.width",
        240,
    ):
        display(frame)


def summarize_report(report_name: str, report: dict) -> dict:
    comparisons = report.get("comparisons", {})
    direction_cosines = report.get("direction_cosines", {})
    perturbation_controls = report.get("perturbation_controls", {})
    pipeline_state_map = globals().get("pipeline_state_reports", {})
    pipeline_state_report = pipeline_state_map.get(report_name, {})
    summary = {
        "report": report_name,
        "mode": report.get("mode"),
        "reference_report_name": report.get("reference_report_name"),
        "embed_vs_store_plain_jaccard": comparisons.get("embed_vs_store_plain", {}).get("jaccard"),
        "embed_vs_store_context_jaccard": comparisons.get("embed_vs_store_context", {}).get("jaccard"),
        "store_plain_vs_context_jaccard": comparisons.get("store_plain_vs_context", {}).get("jaccard"),
        "embed_vs_store_plain_score_cosine": comparisons.get("embed_vs_store_plain", {}).get("shared_score_cosine"),
        "embed_vs_store_context_score_cosine": comparisons.get("embed_vs_store_context", {}).get("shared_score_cosine"),
        "embed_vs_store_plain_direction_cosine": direction_cosines.get("embed_vs_store_plain"),
        "embed_vs_store_context_direction_cosine": direction_cosines.get("embed_vs_store_context"),
        "store_plain_vs_context_direction_cosine": direction_cosines.get("store_plain_vs_context"),
        "embed_random_direction_cosine": perturbation_controls.get("embed", {}).get("direction_cosine_to_base"),
        "store_plain_random_direction_cosine": perturbation_controls.get("store_plain", {}).get(
            "direction_cosine_to_base"
        ),
        "store_context_random_direction_cosine": perturbation_controls.get("store_context", {}).get(
            "direction_cosine_to_base"
        ),
        "has_pipeline_state": report_name in pipeline_state_map,
    }
    for section in ("embed", "store_plain", "store_context"):
        direction_payload = _direction_payload(pipeline_state_report.get(section, {}))
        raw = direction_payload.get("geometry", {}).get("raw_direction_summary", {})
        summary[f"{section}_raw_norm"] = raw.get("norm")
    return summary


def summarize_reference_report(report_name: str, report: dict) -> dict:
    comparisons = report.get("comparisons", {})
    direction_cosines = report.get("direction_cosines", {})
    return {
        "report": report_name,
        "mode": report.get("mode"),
        "calibration_surface": report.get("calibration_surface"),
        "embed_vs_store_plain_jaccard": comparisons.get("embed_vs_store_plain", {}).get("jaccard"),
        "embed_vs_store_context_jaccard": comparisons.get("embed_vs_store_context", {}).get("jaccard"),
        "embed_vs_embed_random_perturbed_jaccard": comparisons.get("embed_vs_embed_random_perturbed", {}).get(
            "jaccard"
        ),
        "store_plain_vs_random_jaccard": comparisons.get("store_plain_vs_store_plain_random_perturbed", {}).get(
            "jaccard"
        ),
        "store_context_vs_random_jaccard": comparisons.get("store_context_vs_store_context_random_perturbed", {}).get(
            "jaccard"
        ),
        "embed_random_direction_cosine": direction_cosines.get("embed_vs_embed_random_perturbed"),
        "store_plain_random_direction_cosine": direction_cosines.get("store_plain_vs_store_plain_random_perturbed"),
        "store_context_random_direction_cosine": direction_cosines.get(
            "store_context_vs_store_context_random_perturbed"
        ),
    }


def summarize_notebook_artifact(artifact_name: str, artifact: dict) -> dict:
    comparison = artifact.get("comparison", {})
    shared_features, _, _ = _feature_partitions(comparison)
    embed_direction = _direction_payload(artifact.get("embed_direction", {}))
    store_direction = _direction_payload(artifact.get("store_direction", {}))
    embed_geometry = embed_direction.get("geometry", {})
    store_geometry = store_direction.get("geometry", {})
    perturbation_controls = artifact.get("perturbation_controls", {})
    embed_control = perturbation_controls.get("embed", {})
    store_control = perturbation_controls.get("store", {})
    return {
        "artifact": artifact_name,
        "embed_raw_norm": embed_geometry.get("raw_direction_summary", {}).get("norm"),
        "store_raw_norm": store_geometry.get("raw_direction_summary", {}).get("norm"),
        "feature_jaccard": comparison.get("feature_jaccard"),
        "cosine_similarity": comparison.get("cosine_similarity"),
        "shared_feature_count": comparison.get("shared_feature_count", len(shared_features)),
        "embed_gap_delta": artifact.get("embed_pipeline", {}).get("gap_delta"),
        "store_gap_delta": artifact.get("store_pipeline", {}).get("gap_delta"),
        "embed_random_direction_cosine": embed_control.get("direction_cosine_to_base"),
        "embed_random_feature_jaccard": embed_control.get("feature_jaccard_vs_base"),
        "store_random_direction_cosine": store_control.get("direction_cosine_to_base"),
        "store_random_feature_jaccard": store_control.get("feature_jaccard_vs_base"),
    }


def build_metrics_frame(report: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "path": label,
                "pre_gap": report[label].get("pre_gap"),
                "post_gap": report[label].get("post_gap"),
                "group_a_projection_mean": report[label].get("group_a_projection_mean"),
                "group_b_projection_mean": report[label].get("group_b_projection_mean"),
                "top_feature_count": len(report[label].get("top_feature_ids", [])),
                "top_feature_ids": _json_cell(report[label].get("top_feature_ids", [])),
                "top_feature_scores": _json_cell(report[label].get("top_feature_scores", [])),
            }
            for label in ("embed", "store_plain", "store_context")
        ]
    )


def build_direction_geometry_frame(
    pipeline_state_report: dict,
    *,
    sections: tuple[str, ...],
) -> pd.DataFrame:
    rows = []
    for section in sections:
        direction_payload = _direction_payload(pipeline_state_report.get(section, {}))
        if not direction_payload:
            continue
        geometry = direction_payload.get("geometry", {})
        raw = geometry.get("raw_direction_summary", geometry.get("raw_direction", {}))
        normalized = geometry.get(
            "normalized_direction_summary",
            geometry.get("normalized_direction", {}),
        )
        rows.append(
            {
                "path": section,
                "raw_norm": raw.get("norm"),
                "normalized_norm": normalized.get("norm"),
                "raw_mean_abs": raw.get("mean_abs"),
                "normalized_mean_abs": normalized.get("mean_abs"),
                "latent_row_count": geometry.get("latent_row_count"),
                "latent_row_norm_head": _json_cell(_head(geometry.get("latent_row_norms", []))),
                "example_weight_head": _json_cell(_head(geometry.get("example_weights", []))),
                "pair_residual_norm_head": _json_cell(_head(geometry.get("pair_residual_norms", []))),
                "raw_sha": _short_sha(raw.get("sha256")),
                "normalized_sha": _short_sha(normalized.get("sha256")),
            }
        )
    return pd.DataFrame(rows)


def build_prompt_alignment_frame(pipeline_state_report: dict, *, section: str) -> pd.DataFrame:
    direction_payload = _direction_payload(pipeline_state_report.get(section, {}))
    prompt_examples = direction_payload.get("prompt_examples", [])
    frame = pd.DataFrame(
        [
            {
                "probe_text": example.get("probe_text"),
                "probe_surface_text": example.get("probe_surface_text"),
                "probe_start_index": example.get("probe_start_index"),
                "probe_end_index": example.get("probe_end_index"),
                "answer_text": example.get("answer_text"),
                "answer_index": example.get("answer_index"),
                "cache_answer_index": example.get("cache_answer_index"),
                "cache_answer_token": _json_cell(
                    [
                        example.get("cache_answer_token_id"),
                        example.get("cache_answer_token_text"),
                    ]
                ),
                "context_token_index": example.get("context_token_index"),
                "context_token": _json_cell(
                    [
                        example.get("context_token_id"),
                        example.get("context_token_text"),
                    ]
                ),
                "context_token_source": example.get("context_token_source"),
                "cache_prompt_token_debug": _format_token_debug_html(example.get("cache_prompt_token_debug", [])),
            }
            for example in prompt_examples
        ]
    )
    frame.attrs["html_columns"] = ("cache_prompt_token_debug",)
    return frame


def build_extraction_frame(pipeline_state_report: dict) -> pd.DataFrame:
    direction_payload = _direction_payload(pipeline_state_report.get("store_context", {}))
    snapshots = direction_payload.get("context_enhanced_extraction", [])
    frame = pd.DataFrame(
        [
            {
                "probe_surface_text": snapshot.get("probe_surface_text"),
                "cache_answer_index": snapshot.get("cache_answer_index"),
                "context_token_index": snapshot.get("context_token_index"),
                "context_token_source": snapshot.get("context_token_source"),
                "answer_indices": _json_cell(snapshot.get("answer_indices")),
                "context_indices": _json_cell(snapshot.get("context_indices")),
                "dot_num": snapshot.get("dot_num"),
                "dot_den": snapshot.get("dot_den"),
                "cache_prompt_token_debug": _format_token_debug_html(snapshot.get("cache_prompt_token_debug", [])),
            }
            for snapshot in snapshots
        ]
    )
    frame.attrs["html_columns"] = ("cache_prompt_token_debug",)
    return frame


def build_graph_stage_frame(
    pipeline_state_report: dict,
    *,
    sections: tuple[str, ...] | None = None,
) -> pd.DataFrame:
    rows = []
    section_names = sections or tuple(_graph_sections(pipeline_state_report))
    for section in section_names:
        graph_payload = _graph_payload(pipeline_state_report, section)
        if not graph_payload:
            continue
        rows.append(
            {
                "path": section,
                "path_label": graph_payload.get("path_label"),
                "graph_input_tokens": _json_cell(graph_payload.get("graph_input_tokens")),
                "top_features": _json_cell(_head(graph_payload.get("top_features", []), size=10)),
                "top_feature_scores": _json_cell(_head(graph_payload.get("top_feature_scores", []), size=10)),
                "pre_gap": graph_payload.get("pre_gap"),
                "post_gap": graph_payload.get("post_gap"),
                "gap_delta": graph_payload.get("gap_delta"),
            }
        )
    return pd.DataFrame(rows)


def build_reference_metrics_frame(report: dict) -> pd.DataFrame:
    labels = _reference_graph_labels(report)
    return pd.DataFrame(
        [
            {
                "path": label,
                "selected_feature_count": report[label].get("selected_feature_count"),
                "active_feature_count": report[label].get("active_feature_count"),
                "top_feature_count": len(report[label].get("top_feature_ids", [])),
                "top_feature_ids": _json_cell(report[label].get("top_feature_ids", [])),
                "top_feature_scores": _json_cell(report[label].get("top_feature_scores", [])),
            }
            for label in labels
        ]
    )


def build_reference_comparison_frame(report: dict) -> pd.DataFrame:
    comparisons = report.get("comparisons", {})
    rows = []
    for label, payload in comparisons.items():
        shared, left_only, right_only = _feature_partitions(payload)
        rows.append(
            {
                "comparison": label,
                "jaccard": payload.get("jaccard"),
                "shared_score_cosine": payload.get("shared_score_cosine"),
                "shared_feature_count": len(shared),
                "left_only_count": len(left_only),
                "right_only_count": len(right_only),
                "shared_features": _json_cell(_head(shared, size=10)),
            }
        )
    return pd.DataFrame(rows)


def build_notebook_pipeline_frame(artifact: dict) -> pd.DataFrame:
    rows = []
    for label, key in (
        ("embed_pipeline", "embed_pipeline"),
        ("store_pipeline", "store_pipeline"),
        ("embed_random_perturbed", "embed_random_perturbed_pipeline"),
        ("store_random_perturbed", "store_random_perturbed_pipeline"),
    ):
        payload = artifact.get(key, {})
        if not payload:
            continue
        rows.append(
            {
                "path": label,
                "pre_gap": payload.get("pre_gap"),
                "post_gap": payload.get("post_gap"),
                "gap_delta": payload.get("gap_delta"),
                "top_features": _json_cell(_head(payload.get("top_features", []), size=10)),
                "top_feature_scores": _json_cell(_head(payload.get("top_feature_scores", []), size=10)),
            }
        )
    return pd.DataFrame(rows)


def build_notebook_perturbation_frame(artifact: dict) -> pd.DataFrame:
    perturbation_controls = artifact.get("perturbation_controls", {})
    rows = []
    for label, payload in perturbation_controls.items():
        rows.append(
            {
                "path": label,
                "direction_cosine_to_base": payload.get("direction_cosine_to_base"),
                "feature_jaccard_vs_base": payload.get("feature_jaccard_vs_base"),
                "shared_score_cosine_vs_base": payload.get("shared_score_cosine_vs_base"),
                "gap_delta_vs_base": payload.get("gap_delta_vs_base"),
            }
        )
    return pd.DataFrame(rows)


def build_notebook_comparison_frame(artifact: dict) -> pd.DataFrame:
    comparison = artifact.get("comparison", {})
    shared_features, embed_only_features, store_only_features = _feature_partitions(comparison)
    return pd.DataFrame(
        [
            {
                "feature_jaccard": comparison.get("feature_jaccard"),
                "cosine_similarity": comparison.get("cosine_similarity"),
                "shared_score_cosine": comparison.get("shared_score_cosine"),
                "shared_feature_count": comparison.get("shared_feature_count", len(shared_features)),
                "embed_only_count": comparison.get("embed_only_count", len(embed_only_features)),
                "store_only_count": comparison.get("store_only_count", len(store_only_features)),
                "shared_features": _json_cell(_head(shared_features, size=10)),
                "embed_only_features": _json_cell(_head(embed_only_features, size=10)),
                "store_only_features": _json_cell(_head(store_only_features, size=10)),
                "shared_key_token_rows": _json_cell(_head(comparison.get("shared_key_token_rows", []), size=10)),
            }
        ]
    )

In [ ]:
from itertools import combinations

NOTEBOOK_REFERENCE_REPORT_OVERRIDES = {
    "gemma3_1b_it_ohio_notebook_debug": "gemma3_1b_it_ohio_reference_graph_sanity_two_group",
    "gemma3_1b_it_orange_notebook_debug": "gemma3_1b_it_orange_reference_graph_sanity",
    "gemma3_1b_it_bat_notebook_debug": "gemma3_1b_it_bat_reference_graph_sanity",
}


def resolve_ohio_reference_report_name(report_name: str) -> str | None:
    if report_name.endswith("single_group"):
        return "gemma3_1b_it_ohio_reference_graph_sanity_single_group"
    if report_name.endswith("two_group"):
        return "gemma3_1b_it_ohio_reference_graph_sanity_two_group"
    return None


def resolve_notebook_reference_report_name(artifact_name: str) -> str | None:
    if artifact_name in NOTEBOOK_REFERENCE_REPORT_OVERRIDES:
        reference_name = NOTEBOOK_REFERENCE_REPORT_OVERRIDES[artifact_name]
        return reference_name if reference_name in reference_reports else None

    if not artifact_name.endswith("_notebook_debug"):
        return None

    base_name = artifact_name.removesuffix("_notebook_debug")
    exact_reference_name = f"{base_name}_reference_graph_sanity"
    return exact_reference_name if exact_reference_name in reference_reports else None


def build_pairwise_feature_comparison_frame(payloads: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for left_label, right_label in combinations(payloads.keys(), 2):
        comparison = _pairwise_feature_comparison(
            payloads[left_label],
            payloads[right_label],
            left_label=left_label,
            right_label=right_label,
        )
        shared, left_only, right_only = _feature_partitions(comparison)
        rows.append(
            {
                "comparison": f"{left_label} vs {right_label}",
                "jaccard": comparison.get("jaccard"),
                "shared_score_cosine": comparison.get("shared_score_cosine"),
                "shared_feature_count": len(shared),
                "left_only_count": len(left_only),
                "right_only_count": len(right_only),
                "shared_features": _json_cell(_head(shared, size=10)),
            }
        )
    return pd.DataFrame(rows)


def build_node_influence_alignment_frame(
    payloads: dict[str, dict],
    *,
    limit: int = 15,
) -> pd.DataFrame:
    ordered_rows: list[tuple[int, int, int]] = []
    seen_rows: set[tuple[int, int, int]] = set()
    score_maps: dict[str, dict[tuple[int, int, int], float]] = {}
    rank_maps: dict[str, dict[tuple[int, int, int], int]] = {}

    for label, payload in payloads.items():
        feature_rows = _feature_rows_from_payload(payload)[:limit]
        feature_scores = _feature_scores_from_payload(payload)[:limit]
        score_maps[label] = {row: float(score) for row, score in zip(feature_rows, feature_scores, strict=False)}
        rank_maps[label] = {row: index + 1 for index, row in enumerate(feature_rows)}
        for row in feature_rows:
            if row in seen_rows:
                continue
            ordered_rows.append(row)
            seen_rows.add(row)

    return pd.DataFrame(
        [
            {
                "feature_row": _json_cell(list(row)),
                **{f"{label}_rank": rank_maps[label].get(row) for label in payloads},
                **{f"{label}_score": score_maps[label].get(row) for label in payloads},
            }
            for row in ordered_rows
        ]
    )


def display_feature_score_triad(title: str, payloads: dict[str, dict]) -> None:
    triad_payloads = {label: payload for label, payload in payloads.items() if payload}
    if len(triad_payloads) < 2:
        display(Markdown(f"#### {title}"))
        display(Markdown("No matching artifacts were found for this comparison."))
        return

    feature_sets = {label: _feature_rows_from_payload(payload)[:10] for label, payload in triad_payloads.items()}
    score_sets = {label: _feature_scores_from_payload(payload)[:10] for label, payload in triad_payloads.items()}

    display(Markdown(f"#### {title}"))
    display_top_features_comparison(
        feature_sets,
        scores_sets=score_sets,
        neuronpedia_model=NEURONPEDIA_MODEL_ID,
        neuronpedia_set=NEURONPEDIA_SOURCE_SET,
    )
    display(Markdown("##### Pairwise Feature Comparison"))
    display_full_frame(build_pairwise_feature_comparison_frame(triad_payloads))
    display(Markdown("##### Node Influence Score Alignment"))
    display_full_frame(build_node_influence_alignment_frame(triad_payloads))

In [ ]:
reports = {path.parent.name: load_concept_direction_parity_report(path) for path in report_paths}
reference_reports = {
    path.parent.name: load_concept_direction_reference_graph_report(path) for path in reference_report_paths
}
pipeline_state_reports = {
    report_name: load_concept_direction_pipeline_state_artifacts(path)
    for report_name, path in pipeline_state_paths.items()
}
notebook_pipeline_state_reports = {
    path.parent.name: load_concept_direction_pipeline_state_artifacts(path) for path in notebook_pipeline_state_paths
}


def _summarize_report_v2(report_name: str, report: dict) -> dict:
    comparisons = report.get("comparisons", {})
    direction_cosines = report.get("direction_cosines", {})
    perturbation_controls = report.get("perturbation_controls", {})
    pipeline_state_report = pipeline_state_reports.get(report_name, {})
    summary = {
        "report": report_name,
        "mode": report.get("mode"),
        "reference_report_name": report.get("reference_report_name"),
        "embed_vs_store_plain_jaccard": comparisons.get("embed_vs_store_plain", {}).get("jaccard"),
        "embed_vs_store_context_jaccard": comparisons.get("embed_vs_store_context", {}).get("jaccard"),
        "store_plain_vs_context_jaccard": comparisons.get("store_plain_vs_context", {}).get("jaccard"),
        "embed_vs_store_plain_score_cosine": comparisons.get("embed_vs_store_plain", {}).get("shared_score_cosine"),
        "embed_vs_store_context_score_cosine": comparisons.get("embed_vs_store_context", {}).get("shared_score_cosine"),
        "embed_vs_store_plain_direction_cosine": direction_cosines.get("embed_vs_store_plain"),
        "embed_vs_store_context_direction_cosine": direction_cosines.get("embed_vs_store_context"),
        "store_plain_vs_context_direction_cosine": direction_cosines.get("store_plain_vs_context"),
        "embed_controlled_perturbation_cosine": perturbation_controls.get("embed", {}).get(
            "direction_cosine_to_base",
            direction_cosines.get("embed_vs_embed_random_perturbed"),
        ),
        "embed_sampled_random_direction_cosine": perturbation_controls.get("embed", {}).get(
            "sampled_random_direction_cosine_to_base"
        ),
        "embed_perturbation_basis_cosine": perturbation_controls.get("embed", {}).get(
            "perturbation_basis_cosine_to_base"
        ),
        "store_plain_controlled_perturbation_cosine": perturbation_controls.get("store_plain", {}).get(
            "direction_cosine_to_base",
            direction_cosines.get("store_plain_vs_store_plain_random_perturbed"),
        ),
        "store_plain_sampled_random_direction_cosine": perturbation_controls.get("store_plain", {}).get(
            "sampled_random_direction_cosine_to_base"
        ),
        "store_plain_perturbation_basis_cosine": perturbation_controls.get("store_plain", {}).get(
            "perturbation_basis_cosine_to_base"
        ),
        "store_context_controlled_perturbation_cosine": perturbation_controls.get("store_context", {}).get(
            "direction_cosine_to_base",
            direction_cosines.get("store_context_vs_store_context_random_perturbed"),
        ),
        "store_context_sampled_random_direction_cosine": perturbation_controls.get("store_context", {}).get(
            "sampled_random_direction_cosine_to_base"
        ),
        "store_context_perturbation_basis_cosine": perturbation_controls.get("store_context", {}).get(
            "perturbation_basis_cosine_to_base"
        ),
        "has_pipeline_state": report_name in pipeline_state_reports,
    }
    for section in ("embed", "store_plain", "store_context"):
        direction_payload = _direction_payload(pipeline_state_report.get(section, {}))
        raw = direction_payload.get("geometry", {}).get("raw_direction_summary", {})
        summary[f"{section}_raw_norm"] = raw.get("norm")
    return summary


def _summarize_reference_report_v2(report_name: str, report: dict) -> dict:
    comparisons = report.get("comparisons", {})
    direction_cosines = report.get("direction_cosines", {})
    return {
        "report": report_name,
        "mode": report.get("mode"),
        "calibration_surface": report.get("calibration_surface"),
        "embed_vs_store_plain_jaccard": comparisons.get("embed_vs_store_plain", {}).get("jaccard"),
        "embed_vs_store_context_jaccard": comparisons.get("embed_vs_store_context", {}).get("jaccard"),
        "embed_vs_embed_random_perturbed_jaccard": comparisons.get("embed_vs_embed_random_perturbed", {}).get(
            "jaccard"
        ),
        "store_plain_vs_random_jaccard": comparisons.get("store_plain_vs_store_plain_random_perturbed", {}).get(
            "jaccard"
        ),
        "store_context_vs_random_jaccard": comparisons.get("store_context_vs_store_context_random_perturbed", {}).get(
            "jaccard"
        ),
        "embed_controlled_perturbation_cosine": direction_cosines.get("embed_vs_embed_random_perturbed"),
        "store_plain_controlled_perturbation_cosine": direction_cosines.get(
            "store_plain_vs_store_plain_random_perturbed"
        ),
        "store_context_controlled_perturbation_cosine": direction_cosines.get(
            "store_context_vs_store_context_random_perturbed"
        ),
    }


def _summarize_notebook_artifact_v2(artifact_name: str, artifact: dict) -> dict:
    comparison = artifact.get("comparison", {})
    shared_features, _, _ = _feature_partitions(comparison)
    embed_direction = _direction_payload(artifact.get("embed_direction", {}))
    store_direction = _direction_payload(artifact.get("store_direction", {}))
    embed_geometry = embed_direction.get("geometry", {})
    store_geometry = store_direction.get("geometry", {})
    perturbation_controls = artifact.get("perturbation_controls", {})
    embed_control = perturbation_controls.get("embed", {})
    store_control = perturbation_controls.get("store", {})
    return {
        "artifact": artifact_name,
        "embed_raw_norm": embed_geometry.get("raw_direction_summary", {}).get("norm"),
        "store_raw_norm": store_geometry.get("raw_direction_summary", {}).get("norm"),
        "feature_jaccard": comparison.get("feature_jaccard"),
        "cosine_similarity": comparison.get("cosine_similarity"),
        "shared_feature_count": comparison.get("shared_feature_count", len(shared_features)),
        "embed_gap_delta": artifact.get("embed_pipeline", {}).get("gap_delta"),
        "store_gap_delta": artifact.get("store_pipeline", {}).get("gap_delta"),
        "embed_controlled_perturbation_cosine": embed_control.get("direction_cosine_to_base"),
        "embed_sampled_random_direction_cosine": embed_control.get("sampled_random_direction_cosine_to_base"),
        "embed_perturbation_basis_cosine": embed_control.get("perturbation_basis_cosine_to_base"),
        "embed_random_feature_jaccard": embed_control.get("feature_jaccard_vs_base"),
        "store_controlled_perturbation_cosine": store_control.get("direction_cosine_to_base"),
        "store_sampled_random_direction_cosine": store_control.get("sampled_random_direction_cosine_to_base"),
        "store_perturbation_basis_cosine": store_control.get("perturbation_basis_cosine_to_base"),
        "store_random_feature_jaccard": store_control.get("feature_jaccard_vs_base"),
    }


summary_frame = (
    pd.DataFrame([_summarize_report_v2(report_name, report) for report_name, report in reports.items()]).sort_values(
        "report"
    )
    if reports
    else pd.DataFrame()
)

reference_summary_frame = (
    pd.DataFrame(
        [_summarize_reference_report_v2(report_name, report) for report_name, report in reference_reports.items()]
    ).sort_values("report")
    if reference_reports
    else pd.DataFrame()
)

notebook_summary_frame = (
    pd.DataFrame(
        [
            _summarize_notebook_artifact_v2(artifact_name, artifact)
            for artifact_name, artifact in notebook_pipeline_state_reports.items()
        ]
    ).sort_values("artifact")
    if notebook_pipeline_state_reports
    else pd.DataFrame()
)

selected_report_name = next(iter(sorted(reports)), None)
selected_reference_report_name = next(iter(sorted(reference_reports)), None)
selected_notebook_artifact_name = next(iter(sorted(notebook_pipeline_state_reports)), None)

## Summary Tables

### Direct-Op Report Index

In [ ]:
if summary_frame.empty:
    display(Markdown("No direct-op parity reports matched the current artifact selection."))
else:
    display_full_frame(summary_frame)

### Reference Graph Sanity Index

In [ ]:
if reference_summary_frame.empty:
    display(Markdown("No reference graph sanity reports matched the current artifact selection."))
else:
    display_full_frame(reference_summary_frame)

### Notebook Debug Artifact Index

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts matched the current artifact selection."))
else:
    display_full_frame(notebook_summary_frame)

## Direct-Op Report Details

### Per-Report Metrics and Top Features

In [ ]:
if not reports:
    display(Markdown("No direct-op parity reports are available."))
else:
    for report_name in sorted(reports):
        report = reports[report_name]
        pipeline_state_report = pipeline_state_reports.get(report_name, {})

        display(Markdown(f"### direct-op::{report_name}"))
        display_full_frame(build_metrics_frame(report))
        display(Markdown("#### Direction Geometry"))
        display_full_frame(
            build_direction_geometry_frame(
                pipeline_state_report,
                sections=("embed", "store_plain", "store_context"),
            )
        )
        display_top_features_comparison(
            {
                "embed": _normalize_feature_rows(report["embed"].get("top_feature_ids", [])),
                "store_plain": _normalize_feature_rows(report["store_plain"].get("top_feature_ids", [])),
                "store_context": _normalize_feature_rows(report["store_context"].get("top_feature_ids", [])),
            },
            scores_sets={
                "embed": report["embed"].get("top_feature_scores"),
                "store_plain": report["store_plain"].get("top_feature_scores"),
                "store_context": report["store_context"].get("top_feature_scores"),
            },
            neuronpedia_model=NEURONPEDIA_MODEL_ID,
            neuronpedia_set=NEURONPEDIA_SOURCE_SET,
        )

### Store Prompt Alignment

In [ ]:
if not reports:
    display(Markdown("No direct-op parity reports are available."))
else:
    for report_name in sorted(reports):
        pipeline_state_report = pipeline_state_reports.get(report_name, {})
        display(Markdown(f"### store-prompt-alignment::{report_name}"))
        display(Markdown("#### store_plain"))
        display_full_frame(
            build_prompt_alignment_frame(
                pipeline_state_report,
                section="store_plain",
            )
        )
        display(Markdown("#### store_context"))
        display_full_frame(
            build_prompt_alignment_frame(
                pipeline_state_report,
                section="store_context",
            )
        )

### Context-Enhanced Extraction

In [ ]:
if not reports:
    display(Markdown("No direct-op parity reports are available."))
else:
    for report_name in sorted(reports):
        pipeline_state_report = pipeline_state_reports.get(report_name, {})
        display(Markdown(f"### context-extraction::{report_name}"))
        display_full_frame(build_extraction_frame(pipeline_state_report))

### Direct-Op Graph Stage

In [ ]:
if not reports:
    display(Markdown("No direct-op parity reports are available."))
else:
    for report_name in sorted(reports):
        pipeline_state_report = pipeline_state_reports.get(report_name, {})
        display(Markdown(f"### graph-stage::{report_name}"))
        display_full_frame(build_graph_stage_frame(pipeline_state_report))

## Reference Graph Sanity Details

### Reference Metrics and Pairwise Comparisons

In [ ]:
if reference_summary_frame.empty:
    display(Markdown("No reference graph sanity reports are available."))
else:
    for report_name in sorted(reference_reports):
        report = reference_reports[report_name]
        display(Markdown(f"### reference::{report_name}"))
        display_full_frame(build_reference_metrics_frame(report))
        display(Markdown("#### Pairwise Reference Comparisons"))
        display_full_frame(build_reference_comparison_frame(report))

### Ohio Direct-Op vs Perturbed vs Reference

In [ ]:
ohio_report_names = sorted(reports)

if not ohio_report_names:
    display(Markdown("No Ohio direct-op reports are available."))
else:
    for report_name in ohio_report_names:
        pipeline_state_report = pipeline_state_reports.get(report_name, {})
        reference_name = resolve_ohio_reference_report_name(report_name)
        reference_report = reference_reports.get(reference_name, {}) if reference_name else {}

        display(Markdown(f"### ohio::{report_name}"))
        if not reference_report:
            display(Markdown("No matching Ohio reference report was found for this direct-op artifact."))
            continue

        display_feature_score_triad(
            "Embed vs Embed Perturbed vs Embed Reference",
            {
                "embed": _graph_payload(pipeline_state_report, "embed"),
                "embed_perturbed": _graph_payload(
                    pipeline_state_report,
                    "embed_random_perturbed",
                ),
                "embed_reference": reference_report.get("embed", {}),
            },
        )
        display_feature_score_triad(
            "Store Context vs Store Context Perturbed vs Store Context Reference",
            {
                "store_context": _graph_payload(pipeline_state_report, "store_context"),
                "store_context_perturbed": _graph_payload(
                    pipeline_state_report,
                    "store_context_random_perturbed",
                ),
                "store_context_reference": reference_report.get("store_context", {}),
            },
        )
        display_feature_score_triad(
            "Store Plain vs Store Plain Perturbed vs Store Plain Reference",
            {
                "store_plain": _graph_payload(pipeline_state_report, "store_plain"),
                "store_plain_perturbed": _graph_payload(
                    pipeline_state_report,
                    "store_plain_random_perturbed",
                ),
                "store_plain_reference": reference_report.get("store_plain", {}),
            },
        )

## Notebook vs Reference Triads

In [ ]:
no_reference_payload = {"top_feature_ids": [], "top_feature_scores": []}

if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        reference_name = resolve_notebook_reference_report_name(artifact_name)
        reference_report = reference_reports.get(reference_name, {}) if reference_name else {}
        has_reference = bool(reference_report)

        display(Markdown(f"### notebook-triads::{artifact_name}"))
        if not has_reference:
            display(
                Markdown("No reference available: no matching reference report was found for this notebook artifact.")
            )

        display_feature_score_triad(
            "Embed Pipeline vs Embed Perturbed vs Embed Reference",
            {
                "embed": artifact.get("embed_pipeline", {}),
                "embed_perturbed": artifact.get("embed_random_perturbed_pipeline", {}),
                "embed_reference" if has_reference else "No reference available": (
                    reference_report.get("embed", {}) if has_reference else no_reference_payload
                ),
            },
        )
        display_feature_score_triad(
            "Store Pipeline vs Store Perturbed vs Store Reference",
            {
                "store": artifact.get("store_pipeline", {}),
                "store_perturbed": artifact.get("store_random_perturbed_pipeline", {}),
                "store_reference" if has_reference else "No reference available": (
                    reference_report.get("store_context", {}) if has_reference else no_reference_payload
                ),
            },
        )

## Notebook Debug Artifact Details

### Direction Geometry

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        display(Markdown(f"### notebook-direction-geometry::{artifact_name}"))
        display_full_frame(
            build_direction_geometry_frame(
                artifact,
                sections=("embed_direction", "store_direction"),
            )
        )

### Store Prompt Alignment

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        display(Markdown(f"### notebook-store-prompt-alignment::{artifact_name}"))
        display_full_frame(
            build_prompt_alignment_frame(
                artifact,
                section="store_direction",
            )
        )

### Notebook Pipeline Outputs

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        display(Markdown(f"### notebook-pipeline::{artifact_name}"))
        display_full_frame(build_notebook_pipeline_frame(artifact))
        display_top_features_comparison(
            {
                "embed": _feature_rows_from_payload(artifact.get("embed_pipeline", {})),
                "store": _feature_rows_from_payload(artifact.get("store_pipeline", {})),
            },
            scores_sets={
                "embed": _feature_scores_from_payload(artifact.get("embed_pipeline", {})),
                "store": _feature_scores_from_payload(artifact.get("store_pipeline", {})),
            },
            neuronpedia_model=NEURONPEDIA_MODEL_ID,
            neuronpedia_set=NEURONPEDIA_SOURCE_SET,
        )

### Random Perturbation Controls

`controlled_perturbation_cosine` is the cosine between the base direction and the fixed-angle perturbed direction.
`sampled_random_direction_cosine` is the cosine between the base direction and the raw sampled random unit vector before orthogonalization.
`perturbation_basis_cosine` should stay near zero when the perturbation basis has been orthogonalized against the base direction.

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        perturbation_controls = artifact.get("perturbation_controls", {})
        frame = pd.DataFrame(
            [
                {
                    "path": label,
                    "controlled_perturbation_cosine": payload.get("direction_cosine_to_base"),
                    "sampled_random_direction_cosine": payload.get("sampled_random_direction_cosine_to_base"),
                    "perturbation_basis_cosine": payload.get("perturbation_basis_cosine_to_base"),
                    "feature_jaccard_vs_base": payload.get("feature_jaccard_vs_base"),
                    "shared_score_cosine_vs_base": payload.get("shared_score_cosine_vs_base"),
                    "gap_delta_vs_base": payload.get("gap_delta_vs_base"),
                }
                for label, payload in perturbation_controls.items()
            ]
        )
        display(Markdown(f"### notebook-perturbation-controls::{artifact_name}"))
        display_full_frame(frame)

### Notebook Comparison Tables

In [ ]:
if notebook_summary_frame.empty:
    display(Markdown("No notebook debug artifacts are available."))
else:
    for artifact_name in sorted(notebook_pipeline_state_reports):
        artifact = notebook_pipeline_state_reports[artifact_name]
        display(Markdown(f"### notebook-comparisons::{artifact_name}"))
        display_full_frame(build_notebook_comparison_frame(artifact))

## Raw Payload Inspection

In [ ]:
raw_payload_bundle = {
    "direct_op_reports": reports,
    "reference_reports": reference_reports,
    "notebook_pipeline_state_reports": notebook_pipeline_state_reports,
}

if not any(raw_payload_bundle.values()):
    display(Markdown("No raw payloads are available for inspection."))
else:
    payload_names = [name for name, payload in raw_payload_bundle.items() if payload]
    selected_payload_name = payload_names[0]
    display(Markdown(f"Showing `{selected_payload_name}`."))
    pprint(raw_payload_bundle[selected_payload_name])